# Model Graphs

Four illustrative figures for the theoretical framework section.

- **Fig 1** `fig_preai.png` — pre-AI wage schedule
- **Fig 2** `fig_postai.png` — post-AI shock: cutoff and wage compression
- **Fig 3** `fig_capability.png` — increasing AI capability: expanding automation frontier
- **Fig 4** `fig_multiple.png` — multiple crossings: automated set as union of intervals

Output to `../output/model/`.

In [26]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import os

os.makedirs('../output/model', exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [27]:
# ── Parameters ────────────────────────────────────────────────────────────────
# p0(i) = a + b*i  (linearly increasing pre-AI task price, for illustration)
alpha = 2.5
a, b  = 0.4, 0.6

def p0(i):
    return a + b * np.asarray(i, dtype=float)

def ai_cost(i, tau):
    return tau * np.exp(alpha * np.asarray(i, dtype=float))

def find_cutoff(tau):
    """Solve tau*exp(alpha*I) = p0(I) for unique I in (0,1)."""
    f = lambda i: ai_cost(i, tau) - p0(i)
    if f(1e-9) >= 0 or f(1 - 1e-9) <= 0:
        return None
    return brentq(f, 1e-9, 1 - 1e-9)

# Verify single-crossing holds for our illustration tau values
for tau in [0.35, 0.25, 0.15]:
    I = find_cutoff(tau)
    print(f'tau={tau:.2f}  ->  I* = {I:.3f}')

tau=0.35  ->  I* = 0.119
tau=0.25  ->  I* = 0.361
tau=0.15  ->  I* = 0.671


In [ ]:
# ── Figure 1: Pre-AI Equilibrium ──────────────────────────────────────────────
i = np.linspace(0, 1, 300)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(i, p0(i), color='#2166ac', lw=2.5,
        label=r'$w_0(i^*) = p_0(i^*)$')
ax.fill_between(i, 0, p0(i), alpha=0.08, color='#2166ac')

ax.set_xlabel(r'Task index / worker type $i^*$', fontsize=12)
ax.set_ylabel('Task price / wage', fontsize=12)
ax.set_title('Pre-AI Equilibrium: Wage Schedule', fontsize=13, pad=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.15)
ax.set_xticks([0, 1])
ax.set_yticks([])
ax.legend(fontsize=11, framealpha=0, loc='upper left')

plt.tight_layout()
plt.savefig('../output/model/fig_preai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_preai.png')

In [ ]:
# ── Figure 2: Post-AI Shock ────────────────────────────────────────────────────
tau_main = 0.25
I_star   = find_cutoff(tau_main)
print(f'I* = {I_star:.3f}')

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(i, p0(i),                color='#2166ac', lw=2.5, ls='--',
        label=r'Pre-AI task price $p_0(i)$')
ax.plot(i, ai_cost(i, tau_main), color='#d73027', lw=2.0, ls=':',
        label=r'AI cost $\tau e^{\alpha i}$')
ax.plot(i, np.minimum(p0(i), ai_cost(i, tau_main)),
        color='#1a9641', lw=2.5,
        label=r'Post-AI price $p(i)$')

# Automated region
ax.axvline(I_star, color='#555555', lw=1.2, ls='--')
ax.fill_betweenx([0, 1.5], 0, I_star, alpha=0.08, color='#d73027')
ax.text(I_star * 0.5, 1.38, 'automated', ha='center',
        fontsize=10, color='#d73027', fontstyle='italic')
ax.annotate(r'$I^*$', xy=(I_star, 0), xytext=(I_star + 0.01, -0.09),
            ha='left', fontsize=13, color='#555555',
            annotation_clip=False)

# Wage compression arrow — label below the green line
i_ex   = 0.12
w_pre  = float(p0(i_ex))
w_post = float(ai_cost(i_ex, tau_main))
ax.annotate('', xy=(i_ex, w_post), xytext=(i_ex, w_pre),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.text(i_ex + 0.02, w_post - 0.07,
        r'$\Delta w < 0$', fontsize=10, va='top', ha='left')

ax.set_xlabel(r'Task index / worker type $i^*$', fontsize=12)
ax.set_ylabel('Task price / wage', fontsize=12)
ax.set_title('Post-AI Shock: Cutoff and Wage Compression', fontsize=13, pad=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.5)
ax.set_xticks([0, 1])
ax.set_yticks([])
ax.legend(fontsize=10, framealpha=0, loc='lower right')

plt.tight_layout()
plt.savefig('../output/model/fig_postai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_postai.png')

In [ ]:
# ── Figure 3: Increasing AI Capability ────────────────────────────────────────
taus        = [0.35, 0.25, 0.15]
tau_colors  = ['#c45000', '#d73027', '#7b0000']
tau_labels  = [
    r'$\tau = 0.35$ (early AI)',
    r'$\tau = 0.25$',
    r'$\tau = 0.15$ (advanced AI)',
]
I_labels    = [r'$I^*_1$', r'$I^*_2$', r'$I^*_3$']

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(i, p0(i), color='#2166ac', lw=2.5,
        label=r'Pre-AI task price $p_0(i)$', zorder=5)

cutoffs = []
for tau, color, lbl in zip(taus, tau_colors, tau_labels):
    ax.plot(i, ai_cost(i, tau), color=color, lw=1.8, ls='--', label=lbl)
    I_k = find_cutoff(tau)
    if I_k is not None:
        cutoffs.append((I_k, color))
        ax.axvline(I_k, color=color, lw=1.0, ls=':', alpha=0.8)

# Label cutoffs below x-axis
for k, (I_k, color) in enumerate(cutoffs):
    ax.annotate(I_labels[k], xy=(I_k, 0), xytext=(I_k, -0.10),
                ha='center', fontsize=11, color=color,
                annotation_clip=False)

# Shade expanding automation region
prev = 0
for k, (I_k, color) in enumerate(cutoffs):
    ax.fill_betweenx([0, 0.42], prev, I_k,
                     alpha=0.10, color=tau_colors[k])
    prev = I_k

# Arrow indicating direction of decreasing tau
ax.annotate('', xy=(0.72, 0.22), xytext=(0.72, 0.78),
            arrowprops=dict(arrowstyle='->', color='#7b0000', lw=1.5))
ax.text(0.74, 0.50, r'decreasing $\tau$',
        fontsize=10, color='#7b0000', va='center')

ax.set_xlabel(r'Task index / worker type $i^*$', fontsize=12)
ax.set_ylabel('Task price / cost', fontsize=12)
ax.set_title(r'Increasing AI Capability: Automation Frontier Expands',
             fontsize=13, pad=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.5)
ax.set_xticks([0, 1])
ax.set_yticks([])
ax.legend(fontsize=10, framealpha=0, loc='upper left')

plt.tight_layout()
plt.savefig('../output/model/fig_capability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_capability.png')

In [ ]:
# ── Figure 4: Multiple Crossings ──────────────────────────────────────────────
alpha_mc = 2.5
tau_mc   = 0.25

def p0_mc(i):
    i = np.asarray(i, dtype=float)
    return 0.6 - 0.3*i + 0.8*np.exp(-((i - 0.6) / 0.15)**2)

def phi_mc(i):
    i = np.asarray(i, dtype=float)
    return tau_mc * np.exp(alpha_mc * i) - p0_mc(i)

i_plot = np.linspace(0, 1, 500)

# Find all zero crossings numerically
i_dense  = np.linspace(1e-4, 1 - 1e-4, 5000)
phi_vals = phi_mc(i_dense)
sign_idx = np.where(np.diff(np.sign(phi_vals)))[0]
crossings = [brentq(phi_mc, i_dense[k], i_dense[k+1]) for k in sign_idx]
print(f'Crossings at: {[f"{c:.3f}" for c in crossings]}')

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(i_plot, p0_mc(i_plot),                          color='#2166ac', lw=2.5, ls='--',
        label=r'Pre-AI task price $p_0(i)$')
ax.plot(i_plot, tau_mc * np.exp(alpha_mc * i_plot),     color='#d73027', lw=2.0, ls=':',
        label=r'AI cost $\tau e^{\alpha i}$')
ax.plot(i_plot, np.minimum(p0_mc(i_plot),
                            tau_mc * np.exp(alpha_mc * i_plot)),
        color='#1a9641', lw=2.5,
        label=r'Post-AI price $p(i)$')

# Shade automated intervals (where phi < 0)
bounds = [0.0] + crossings + [1.0]
for j in range(len(bounds) - 1):
    i_mid = (bounds[j] + bounds[j+1]) / 2
    if phi_mc(i_mid) < 0:
        ax.fill_betweenx([0, 1.45], bounds[j], bounds[j+1],
                         alpha=0.10, color='#d73027')
        ax.text(i_mid, 0.07, 'auto.',
                ha='center', fontsize=9, color='#d73027', fontstyle='italic')

# Vertical lines and crossing labels below axis
cross_labels = [r'$I^*_1$', r'$I^*_2$', r'$I^*_3$']
for k, cx in enumerate(crossings):
    ax.axvline(cx, color='#555555', lw=1.0, ls=':', alpha=0.7)
    ax.annotate(cross_labels[k], xy=(cx, 0), xytext=(cx, -0.09),
                ha='center', fontsize=11, color='#555555',
                annotation_clip=False)

ax.set_xlabel(r'Task index / worker type $i^*$', fontsize=12)
ax.set_ylabel('Task price / wage', fontsize=12)
ax.set_title(r'Multiple Crossings: $\mathcal{A}$ as a Union of Intervals',
             fontsize=13, pad=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.5)
ax.set_xticks([0, 1])
ax.set_yticks([])
ax.legend(fontsize=10, framealpha=0, loc='upper left')

plt.tight_layout()
plt.savefig('../output/model/fig_multiple.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_multiple.png')